# Countries and Emissions

### Imports and preprocessing

In [1]:
import sys
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

ROOT_DIR = Path.cwd().parent
Tables_DIR = ROOT_DIR / "data/tables"

# helper that saves each graph + summary so src/make_pdf.py can build the PDF
sys.path.insert(0, str(ROOT_DIR / "src"))
from report import save_report

TOPIC = "countries_emissions"

country_income = pd.read_csv(Tables_DIR / "country_ref.csv")
country_gasesdf = pd.read_csv(Tables_DIR / "country_yearly.csv")

# Real countries only (drop World / regions)
df = country_gasesdf.copy()
df = df[df["iso_code"].notna() & ~df["iso_code"].astype(str).str.startswith("OWID")]

# Latest year with broad CO2 coverage (avoids a half-empty final year)
cover = df.groupby("year")["co2_mt"].count()
LATEST = int(cover[cover >= 100].index.max())
latest = df[df["year"] == LATEST].copy()
print("Latest year used:", LATEST, "| countries:", latest["co2_mt"].notna().sum())

BLUE, RED, GREEN, GREY = "#4C78A8", "#E45756", "#54A24B", "#9aa5b1"

def style(fig, title, sub, xt="", yt="", h=480, left=70, legend=True):
    fig.update_layout(
        template="plotly_white", width=1090,
        title=dict(text=f"<b>{title}</b><br><sup>{sub}</sup>",
                   x=0.02, xanchor="left", font=dict(color="#1f3b5c", size=20)),
        xaxis=dict(title=xt, gridcolor="#e8ecf3"),
        yaxis=dict(title=yt, gridcolor="#e8ecf3"),
        legend=dict(orientation="h", x=0.5, xanchor="center", y=-0.2, title_text=""),
        showlegend=legend, height=h, margin=dict(l=left, r=40, t=90, b=90),
    )
    return fig

Latest year used: 2024 | countries: 215


### Q1. Which countries emit the most today?

In [2]:
top = (latest.dropna(subset=["co2_mt"])
             .nlargest(10, "co2_mt")[["country", "co2_mt", "share_global_co2"]]
             .sort_values("co2_mt"))
top10_share = top["share_global_co2"].sum()

# ---- graph ----
fig = go.Figure(go.Bar(
    y=top["country"], x=top["co2_mt"], orientation="h",
    marker_color=[RED if c == top["country"].iloc[-1] else BLUE for c in top["country"]],
    text=[f"{v:,.0f} Mt" for v in top["co2_mt"]], textposition="outside",
    hovertemplate="%{y}: %{x:,.0f} Mt<extra></extra>",
))
style(fig, f"Top 10 CO2 Emitting Countries, {LATEST}",
      f"Top 10 together produce {top10_share:.0f}% of global CO2",
      xt="CO2 emissions (Mt)", legend=False, h=480, left=120)
fig.update_xaxes(range=[0, top["co2_mt"].max() * 1.2])
fig.show()

t = top.sort_values("co2_mt", ascending=False)
print(f"--- Top 10 emitters, {LATEST} ---")
print(t.round(1).to_string(index=False))

# ---- summary ----
c, v, s = t["country"].tolist(), t["co2_mt"].tolist(), t["share_global_co2"].tolist()
summary = (
    f"Emissions are highly concentrated. {c[0]} emits {v[0]:,.0f} Mt, or {s[0]:.1f}% of the global total. "
    f"{c[1]} is second at {v[1]:,.0f} Mt ({s[1]:.1f}%) and {c[2]} third at {v[2]:,.0f} Mt ({s[2]:.1f}%). "
    f"Together these three emit about {sum(s[:3]):.0f}% of world CO2. The top 10 countries produce "
    f"{top10_share:.1f}%, so action by a small group of countries has an outsized effect on global totals."
)
print(summary)
save_report(TOPIC, "q1", "Which countries emit the most today?", fig, summary)

--- Top 10 emitters, 2024 ---
      country  co2_mt  share_global_co2
        China 12289.0              31.8
United States  4904.1              12.7
        India  3193.5               8.3
       Russia  1780.5               4.6
        Japan   961.9               2.5
    Indonesia   812.2               2.1
         Iran   792.6               2.1
 Saudi Arabia   692.1               1.8
  South Korea   583.7               1.5
      Germany   572.3               1.5
Emissions are highly concentrated. China emits 12,289 Mt, or 31.8% of the global total. United States is second at 4,904 Mt (12.7%) and India third at 3,193 Mt (8.3%). Together these three emit about 53% of world CO2. The top 10 countries produce 68.9%, so action by a small group of countries has an outsized effect on global totals.
[report] saved countries_emissions/q1 (1 graph(s))


### Q2. Which countries contributed the most cumulatively since 1850?

In [3]:
cum = df.sort_values("year").groupby("country")["cumulative_co2"].last().dropna()
cum_top = cum.nlargest(10).sort_values()
world_cum = cum.sum()
cum_share = cum_top / world_cum * 100

# ---- graph ----
fig = go.Figure(go.Bar(
    y=cum_top.index, x=cum_top.values, orientation="h",
    marker_color=[RED if c == cum_top.index[-1] else BLUE for c in cum_top.index],
    text=[f"{v:,.0f} Mt ({s:.1f}%)" for v, s in zip(cum_top.values, cum_share.values)],
    textposition="outside", hovertemplate="%{y}: %{x:,.0f} Mt<extra></extra>",
))
style(fig, "Cumulative CO2 Emissions Since 1850",
      f"Top 10 countries account for {cum_share.sum():.0f}% of all historical CO2",
      xt="Cumulative CO2 (Mt)", legend=False, h=480, left=120)
fig.update_xaxes(range=[0, cum_top.max() * 1.3])
fig.show()

print("--- Cumulative CO2 since 1850 (top 10) ---")
print(pd.DataFrame({"cum Mt": cum_top.round(0), "share of total %": cum_share.round(1)})
        .sort_values("cum Mt", ascending=False))

# ---- summary ----
ct = cum_top.sort_values(ascending=False)
cs = cum_share.loc[ct.index]
names = ct.index.tolist()
today3 = latest.nlargest(3, "co2_mt")["country"].tolist()
hist_rank = cum.rank(ascending=False).astype(int)
rank_txt = ", ".join(f"{c} ranks #{hist_rank[c]} ({cum[c] / world_cum * 100:.1f}%)"
                     for c in today3 if c in hist_rank.index)
summary = (
    f"History tells a different story from today. {names[0]} leads with {ct.iloc[0]:,.0f} Mt "
    f"({cs.iloc[0]:.1f}% of all historical CO2), ahead of {names[1]} at {ct.iloc[1]:,.0f} Mt "
    f"({cs.iloc[1]:.1f}%). {names[2]} ({cs.iloc[2]:.1f}%), {names[3]} ({cs.iloc[3]:.1f}%) and {names[4]} "
    f"({cs.iloc[4]:.1f}%) follow, which shows the lasting weight of early industrializers. Among today's three "
    f"largest emitters, {rank_txt} historically. The top 10 account for {cs.sum():.1f}% of all cumulative "
    f"emissions, and {names[0]} carries roughly {ct.iloc[0] / ct.iloc[1]:.1f} times the historical emissions of {names[1]}."
)
print(summary)
save_report(TOPIC, "q2", "Which countries contributed the most cumulatively since 1850?", fig, summary)

--- Cumulative CO2 since 1850 (top 10) ---
                  cum Mt  share of total %
country                                   
United States   434867.0              24.1
China           285087.0              15.8
Russia          122808.0               6.8
Germany          95136.0               5.3
United Kingdom   80079.0               4.4
Japan            69612.0               3.9
India            66073.0               3.7
France           40048.0               2.2
Canada           35644.0               2.0
Ukraine          31236.0               1.7
History tells a different story from today. United States leads with 434,867 Mt (24.1% of all historical CO2), ahead of China at 285,087 Mt (15.8%). Russia (6.8%), Germany (5.3%) and United Kingdom (4.4%) follow, which shows the lasting weight of early industrializers. Among today's three largest emitters, China ranks #2 (15.8%), United States ranks #1 (24.1%), India ranks #7 (3.7%) historically. The top 10 account for 69.9% of all cumul

### Q3. Who emits the most per person?

In [4]:
# Exclude tiny states (under 1M people) so the ranking is meaningful
pc = latest[(latest["population"] >= 1_000_000) & latest["co2_per_capita"].notna()]
pc_top = pc.nlargest(10, "co2_per_capita")[["country", "co2_per_capita", "population"]].sort_values("co2_per_capita")
world_avg = latest["co2_mt"].sum() * 1e6 / latest["population"].sum()   # tonnes per person

# ---- graph ----
fig = go.Figure(go.Bar(
    y=pc_top["country"], x=pc_top["co2_per_capita"], orientation="h",
    marker_color=[RED if c == pc_top["country"].iloc[-1] else BLUE for c in pc_top["country"]],
    text=[f"{v:.1f} t" for v in pc_top["co2_per_capita"]], textposition="outside",
    hovertemplate="%{y}: %{x:.1f} t per person<extra></extra>",
))
fig.add_vline(x=world_avg, line_dash="dash", line_color=GREY,
              annotation_text=f"World avg {world_avg:.1f} t", annotation_position="bottom right")
style(fig, f"Highest CO2 Emissions Per Person, {LATEST}",
      f"Countries with 1M+ people. Top is {pc_top['co2_per_capita'].max() / world_avg:.1f}x the world average",
      xt="CO2 per person (tonnes)", legend=False, h=480, left=120)
fig.update_xaxes(range=[0, pc_top["co2_per_capita"].max() * 1.2])
fig.show()

pcd = pc_top.sort_values("co2_per_capita", ascending=False)
print(f"--- Top 10 per-capita emitters, {LATEST} (population 1M+) ---")
print(pcd.round(2).to_string(index=False))
print(f"\nWorld average (t per person): {world_avg:.2f}")
big = latest.nlargest(6, "co2_mt")[["country", "co2_per_capita"]]
print("\nPer-capita for the 6 largest total emitters:")
print(big.round(2).to_string(index=False))

# ---- summary ----
lead_txt = ", ".join(f"{c} ({v:.1f} t)" for c, v in zip(pcd["country"].head(6), pcd["co2_per_capita"].head(6)))
big_txt = ", ".join(f"{c} {v:.1f} t" for c, v in zip(big["country"], big["co2_per_capita"]))
summary = (
    f"The highest per-person emitters among countries with 1M+ people are {lead_txt}. The leader, "
    f"{pcd['country'].iloc[0]}, emits {pcd['co2_per_capita'].iloc[0] / world_avg:.1f} times the world average of "
    f"{world_avg:.2f} tonnes per person. Among the six largest total emitters, per-person levels are: {big_txt}. "
    f"Total and per-person rankings therefore tell very different stories: a country can dominate world totals "
    f"while its per-person level is moderate, or the reverse."
)
print(summary)
save_report(TOPIC, "q3", "Who emits the most per person?", fig, summary)

--- Top 10 per-capita emitters, 2024 (population 1M+) ---
             country  co2_per_capita  population
               Qatar           41.27   3048429.0
              Kuwait           26.25   4934508.0
             Bahrain           24.27   1607059.0
 Trinidad and Tobago           22.93   1507778.0
        Saudi Arabia           20.38  33962751.0
United Arab Emirates           20.13  11027135.0
                Oman           15.65   5281542.0
           Australia           14.48  26713206.0
       United States           14.20 345426566.0
          Kazakhstan           13.94  20592574.0

World average (t per person): 4.59

Per-capita for the 6 largest total emitters:
      country  co2_per_capita
        China            8.66
United States           14.20
        India            2.20
       Russia           12.30
        Japan            7.77
    Indonesia            2.86
The highest per-person emitters among countries with 1M+ people are Qatar (41.3 t), Kuwait (26.2 t), Bahrain (2

### Q4. Which countries decoupled emissions from GDP growth?

In [5]:
# GDP usually stops before CO2, so find the latest year where BOTH have coverage
cov = df.groupby("year")[["gdp", "co2_mt"]].count()
both = cov[(cov["gdp"] >= 100) & (cov["co2_mt"] >= 100)]
Y1 = int(both.index.max())
Y0 = Y1 - 10
print(f"Decoupling window: {Y0} -> {Y1}")

a = df[df["year"] == Y1][["country", "population", "gdp", "co2_mt"]]
b = df[df["year"] == Y0][["country", "gdp", "co2_mt"]]
d = a.merge(b, on="country", suffixes=("", "_old"))
d = d[d["population"] >= 1_000_000].dropna(subset=["gdp", "gdp_old", "co2_mt", "co2_mt_old"])
d = d[(d["gdp_old"] > 0) & (d["co2_mt_old"] > 0)].copy()

d["gdp_change_10yr_pct"] = (d["gdp"] / d["gdp_old"] - 1) * 100
d["co2_change_10yr_pct"] = (d["co2_mt"] / d["co2_mt_old"] - 1) * 100
d["decoupled"] = (d["gdp_change_10yr_pct"] > 0) & (d["co2_change_10yr_pct"] < 0)

# ---- graph ----
fig = go.Figure()
for flag, color, name in [(False, GREY, "Not decoupled"), (True, GREEN, "Decoupled")]:
    sub = d[d["decoupled"] == flag]
    fig.add_trace(go.Scatter(
        x=sub["co2_change_10yr_pct"], y=sub["gdp_change_10yr_pct"], mode="markers", name=name,
        marker=dict(color=color, size=8, opacity=0.75), text=sub["country"],
        hovertemplate="%{text}<br>CO2: %{x:.1f}%<br>GDP: %{y:.1f}%<extra></extra>",
    ))
lab = d.nlargest(8, "co2_mt")
fig.add_trace(go.Scatter(
    x=lab["co2_change_10yr_pct"], y=lab["gdp_change_10yr_pct"], mode="markers+text",
    text=lab["country"], textposition="top center", showlegend=False, hoverinfo="skip",
    marker=dict(color="#222", size=9),
))
fig.add_vline(x=0, line_color="#333", line_width=1)
style(fig, f"Decoupling: GDP Growth vs CO2 Change, {Y0}-{Y1}",
      f"{d['decoupled'].sum()} of {len(d)} countries grew GDP while cutting CO2 (green, upper left)",
      xt="CO2 change over 10 years (%)", yt="GDP change over 10 years (%)", h=520)
fig.show()

cols4 = ["country", "gdp_change_10yr_pct", "co2_change_10yr_pct"]
print(f"--- Decoupled: {d['decoupled'].sum()} of {len(d)} countries (pop 1M+), {Y0}-{Y1} ---")
print("\nLargest decoupled emitters:")
print(d[d["decoupled"]].nlargest(12, "co2_mt")[cols4].round(1).to_string(index=False))
print("\nBest decoupling (biggest CO2 cut among decoupled):")
print(d[d["decoupled"]].nsmallest(10, "co2_change_10yr_pct")[cols4].round(1).to_string(index=False))
print("\nTop 8 emitters NOT decoupled:")
print(d[~d["decoupled"]].nlargest(8, "co2_mt")[cols4].round(1).to_string(index=False))

# ---- summary ----
def fmt(frame):
    return ", ".join(f"{r.country} (GDP {r.gdp_change_10yr_pct:+.1f}%, CO2 {r.co2_change_10yr_pct:+.1f}%)"
                     for r in frame.itertuples())

n_dec = int(d["decoupled"].sum())
summary = (
    f"Decoupling is real but not universal: {n_dec} of {len(d)} countries (about {n_dec / len(d) * 100:.0f}%) "
    f"grew GDP while cutting CO2 between {Y0} and {Y1}. Large emitters on the list include "
    f"{fmt(d[d['decoupled']].nlargest(5, 'co2_mt'))}. The strongest cuts came from "
    f"{fmt(d[d['decoupled']].nsmallest(4, 'co2_change_10yr_pct'))}. Among the largest emitters that did not "
    f"decouple are {fmt(d[~d['decoupled']].nlargest(4, 'co2_mt'))}. Some decouplers cut emissions only "
    f"marginally, so a decoupled flag does not always mean meaningful reductions."
)
print(summary)
save_report(TOPIC, "q4", "Which countries decoupled emissions from GDP growth?", fig, summary)

Decoupling window: 2012 -> 2022


--- Decoupled: 52 of 153 countries (pop 1M+), 2012-2022 ---

Largest decoupled emitters:
      country  gdp_change_10yr_pct  co2_change_10yr_pct
United States                 23.1                 -5.2
       Russia                  8.5                 -1.4
        Japan                  5.4                -20.9
      Germany                 12.3                -18.0
  South Korea                 29.7                 -3.4
       Canada                 19.0                 -2.5
       Brazil                  5.4                 -3.5
       Mexico                 14.6                 -9.5
 South Africa                  9.5                 -7.1
    Australia                 26.5                 -5.2
        Italy                  4.3                -15.7
       Poland                 43.8                 -3.3

Best decoupling (biggest CO2 cut among decoupled):
       country  gdp_change_10yr_pct  co2_change_10yr_pct
        Greece                  6.5                -37.0
United Kingdom   

### Q5. Which regions and income groups changed their share most over 1990 to now?

In [6]:
def find(*keys):
    for c in country_income.columns:
        if any(k in c.lower() for k in keys):
            return c
    return None

KEY = "iso_code" if "iso_code" in country_income.columns else find("iso", "code")
REGION = find("region")
INCOME = find("income")
print("Using ->", "key:", KEY, "| region:", REGION, "| income:", INCOME)

# One row per country (in case the table repeats countries by year)
meta = (country_income[[KEY, REGION, INCOME]].dropna(subset=[KEY])
        .drop_duplicates(subset=[KEY], keep="last"))
m = df.merge(meta, left_on="iso_code", right_on=KEY, how="left")
Y_START, Y_END = 1990, LATEST

def share_change(group_col):
    out = {}
    for y in (Y_START, Y_END):
        s_ = m[m["year"] == y].dropna(subset=["co2_mt", group_col])
        g_ = s_.groupby(group_col)["co2_mt"].sum()
        out[y] = g_ / g_.sum() * 100
    t_ = pd.DataFrame({f"{Y_START} %": out[Y_START], f"{Y_END} %": out[Y_END]}).fillna(0)
    t_["change (pp)"] = t_[f"{Y_END} %"] - t_[f"{Y_START} %"]
    return t_.sort_values("change (pp)")

reg = share_change(REGION)
inc = share_change(INCOME)

# ---- graph ----
fig = make_subplots(rows=1, cols=2, subplot_titles=("By region", "By income group"), horizontal_spacing=0.22)
for i, t_ in enumerate([reg, inc], start=1):
    fig.add_trace(go.Bar(
        y=t_.index, x=t_["change (pp)"], orientation="h",
        marker_color=[RED if v > 0 else BLUE for v in t_["change (pp)"]],
        text=[f"{v:+.1f} pp" for v in t_["change (pp)"]], textposition="outside", showlegend=False,
        hovertemplate="%{y}: %{x:+.1f} pp<extra></extra>",
    ), row=1, col=i)
    pad = t_["change (pp)"].abs().max() * 1.4
    fig.update_xaxes(range=[-pad, pad], title_text="Change in share of global CO2 (pp)",
                     gridcolor="#e8ecf3", zeroline=True, zerolinecolor="#333", row=1, col=i)
top_r, bot_r = reg["change (pp)"].idxmax(), reg["change (pp)"].idxmin()
fig.update_layout(
    template="plotly_white", width=1090, height=460, margin=dict(l=40, r=40, t=100, b=60),
    title=dict(text=f"<b>Who Gained and Lost Share of Global CO2, {Y_START}-{Y_END}</b>"
                    f"<br><sup>{top_r} gained the most ({reg.loc[top_r, 'change (pp)']:+.1f} pp), "
                    f"{bot_r} lost the most ({reg.loc[bot_r, 'change (pp)']:+.1f} pp)</sup>",
               x=0.02, xanchor="left", font=dict(color="#1f3b5c", size=20)),
)
fig.show()

print(f"--- Share of global CO2 by region, {Y_START} vs {Y_END} ---")
print(reg.sort_values("change (pp)", ascending=False).round(1))
print(f"\n--- Share of global CO2 by income group, {Y_START} vs {Y_END} ---")
print(inc.sort_values("change (pp)", ascending=False).round(1))
unmatched = m[(m["year"] == Y_END) & m[REGION].isna() & m["co2_mt"].notna()]["country"].tolist()
print("\nCountries with CO2 but no region match:", unmatched[:15])

# ---- summary ----
def moves(t_, n=3):
    def f(frame):
        return ", ".join(f"{i} ({r[f'{Y_START} %']:.1f}% to {r[f'{Y_END} %']:.1f}%, {r['change (pp)']:+.1f} pp)"
                         for i, r in frame.iterrows())
    up = t_[t_["change (pp)"] > 0].sort_values("change (pp)", ascending=False).head(n)
    down = t_[t_["change (pp)"] < 0].sort_values("change (pp)").head(n)
    return f(up), f(down)

r_up, r_down = moves(reg)
i_up, i_down = moves(inc)
summary = (
    f"The centre of gravity of global emissions has shifted between {Y_START} and {Y_END}. By region, the biggest "
    f"gains were {r_up}; the biggest losses were {r_down}. By income group, the gains were {i_up}, while the "
    f"losses were {i_down}."
)
print(summary)
save_report(TOPIC, "q5", "Which regions and income groups changed their share most over 1990 to now?", fig, summary)

Using -> key: iso_code | region: region | income: income_group


--- Share of global CO2 by region, 1990 vs 2024 ---
                                                   1990 %  2024 %  change (pp)
region                                                                        
East Asia & Pacific                                  21.6    44.4         22.8
South Asia                                            2.7     9.0          6.3
Middle East, North Africa, Afghanistan & Pakistan     4.4     8.8          4.4
Sub-Saharan Africa                                    2.0     2.4          0.4
Latin America & Caribbean                             4.6     4.7          0.2
North America                                        25.3    14.6        -10.7
Europe & Central Asia                                39.4    16.1        -23.4

--- Share of global CO2 by income group, 1990 vs 2024 ---
                     1990 %  2024 %  change (pp)
income_group                                    
Upper middle income    26.5    50.4         23.9
Lower middle income     5.2    